### Configuration, inputs & outputs

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from pipeline_utils import make_case3_paths, binarize_mutation_inclusive, matrix_stats

paths = make_case3_paths()

## Input files (from 0_xevaset_extraction_summary/3-mcgill_breast_extraction.R)
response_file = paths["datasets"] / "mcgill_breast" / "batch_sensitivity.csv"
model_sensitivity_file = paths["datasets"] / "mcgill_breast" / "model_sensitivity.csv"
expdesign_file = paths["datasets"] / "mcgill_breast" / "expDesign.csv"
omicsmap = paths["datasets"] / "mcgill_breast" / "modToBiobaseMap.csv"
mutation_file = paths["datasets"] / "mcgill_breast" / "mutation_matrix.csv"
cnv_file = paths["datasets"] / "mcgill_breast" / "CNV_matrix.csv"

## Output files
models_out = paths["proc"] / "McGill_cisplatin_models.tsv"
mut_out = paths["proc"] / "McGill_mutation_cisplatin_filtered.csv"
cnv_out = paths["proc"] / "McGill_cnv_cisplatin_filtered.csv"

## NestVNN files
nest_dir = paths["proc"] / "McGill_nest_input"
nest_dir.mkdir(parents=True, exist_ok=True)
gene_file = paths["raw"] / "nest_vnn" / "gene2ind.txt"
mut_vnn = nest_dir / "cell2mutation.txt"
cna_vnn = nest_dir / "cell2amplification.txt"
cnd_vnn = nest_dir / "cell2cndeletion.txt"
cell_vnn = nest_dir / "cell2ind.txt"
gene_vnn = nest_dir / "gene2ind.txt"
test_data_vnn = nest_dir / "test_data.txt"

### Load response table

In [2]:
resp = pd.read_csv(response_file)

print("Total experiments:", resp.shape[0])

Total experiments: 135


### Parse model and drug


In [3]:
# batch.name format is "<model_id>.<drug>" (one row per experimental batch)
resp[["model_id", "drug"]] = resp["batch.name"].str.split(".", expand=True)

n_unparsed = resp[["model_id", "drug"]].isna().any(axis=1).sum()
print("Rows with unparseable batch.name:", n_unparsed)
print("Unique drugs:", resp["drug"].unique())

Rows with unparseable batch.name: 0
Unique drugs: ['Cisplatin' 'Doxorubicin' 'Gemcitabine' 'Paclitaxel' 'Carboplatin']


### Filter cisplatin/carboplatin experiments

In [4]:
cis = resp[resp["drug"].str.lower() == "cisplatin"].copy()
car = resp[resp["drug"].str.lower() == "carboplatin"].copy()
print("Cisplatin experiments:", cis.shape[0])
print("Carboplatin experiments:", car.shape[0])

Cisplatin experiments: 32
Carboplatin experiments: 13


Carboplatin data is too low (n=13) for robust ML model testing.

Will focus on cisplatin for model testing.

### Map cisplatin models to CNV-bearing omics samples

In [5]:
models = sorted(cis["model_id"].unique())
print("Unique cisplatin models:", len(models))

# Xeva model.id in modToBiobaseMap.csv follows "<patient>.<passage>.<GCRC-number><drug-code>",
# e.g. "m5688.12.1715Ci" -> drug-code "Ci" = Cisplatin. Reconstruct "GCRC<number>" to match
# the model_id parsed from batch.name above (e.g. "GCRC1715").
omics = pd.read_csv(omicsmap)
omics = omics[omics["source"] == "PDX"]
omics = omics[omics["model.id"].str.lower().str.endswith("ci")]
omics["model_id"] = "GCRC" + omics["model.id"].str.split(".").str[2].str.replace("Ci", "", regex=False)

# Collapse to one row per (model, biobase sample): a sample can appear once per
# molecular data type (RNAseq/mutation/CNV/logCNV) -- join the types it has.
omics = (
    omics
    .groupby(["model.id", "biobase.id", "source", "model_id"], as_index=False)
    .agg({"mDataType": lambda x: "/".join(sorted(x.unique()))})
)

# Keep only samples with an actual CNV call (exact token match: "logCNV" alone doesn't count)
omics = omics[omics["mDataType"].str.split("/").apply(lambda types: "CNV" in types)].reset_index(drop=True)

omics = omics[["model_id", "biobase.id"]].rename(columns={"biobase.id": "omics_id"})

# Some models resolve to more than one CNV-bearing biobase sample (e.g. multiple
# xenograft passages). There's no way to tell which is authoritative from this
# data alone, so exclude those models entirely rather than guessing.
ambiguous = sorted(omics.loc[omics["model_id"].duplicated(keep=False), "model_id"].unique())
if ambiguous:
    print(f"Cisplatin models excluded (multiple CNV-bearing omics samples, ambiguous): {ambiguous}")
    omics = omics[~omics["model_id"].isin(ambiguous)].reset_index(drop=True)

missing = sorted(set(models) - set(omics["model_id"]))
print(f"Cisplatin models with a CNV-bearing omics sample: {omics.shape[0]}/{len(models)}")
print(f"Cisplatin models excluded (no CNV omics data): {missing}")
print(omics)

Unique cisplatin models: 32
Cisplatin models excluded (multiple CNV-bearing omics samples, ambiguous): ['GCRC1784']
Cisplatin models with a CNV-bearing omics sample: 29/32
Cisplatin models excluded (no CNV omics data): ['GCRC1784', 'GCRC1986', 'GCRC2097']
    model_id      omics_id
0   GCRC1834   SA1028X3_3L
1   GCRC1840         SA998
2   GCRC2029   SA1037X1_2L
3   GCRC1828   SA1027X1_1L
4   GCRC1905         SA997
5   GCRC1945   SA1033X1_2R
6   GCRC1735   SA920X2_23R
7   GCRC1738  SA1030X2_17L
8   GCRC2076    SA1041X2_1
9   GCRC1886   SA1022X1_3L
10  GCRC1851    SA994X2_3L
11  GCRC2006   SA1035X1_8L
12  GCRC1971   SA1019X1_1L
13  GCRC1991   SA1020X1_2L
14  GCRC1979   SA1032X1_2R
15  GCRC1876   SA1017X1_4L
16  GCRC1915   SA1023X1_2L
17  GCRC1924    SA993X2_5R
18  GCRC1939   SA1024X1_1L
19  GCRC1868   SA1016X3_3L
20  GCRC2001   SA1026X1_4R
21  GCRC1863   SA1025X1_2R
22  GCRC1882   SA1021X2_1L
23  GCRC1944   SA1031X1_1R
24  GCRC1963    SA992X1_4R
25  GCRC2061   SA1040X1_1L
26  GCRC2047   

### Compute per-batch n_replicates and majority-voted mRECIST

In [6]:
model_sen = pd.read_csv(model_sensitivity_file)
exp_dsg = pd.read_csv(expdesign_file)

treatment_arm = exp_dsg[exp_dsg["arm"] == "treatment"]

# n_replicates: number of treatment-arm model.id entries recorded per batch in
# expDesign.csv (same definition used for the UHN cohort).
n_replicates_derived = (
    treatment_arm.groupby("batch.name").size().rename("n_replicates").reset_index()
)

# model_sensitivity.csv records mRECIST per treatment-arm model.id (usually one
# per batch, but some batches have multiple, e.g. repeat/rechallenge arms).
# Aggregate to one value per batch via majority vote across that batch's
# treatment-arm model.id entries; ties broken by worst response (PD > SD > PR > CR).
MRECIST_WORST_FIRST = ["PD", "SD", "PR", "CR"]

def majority_vote_mrecist(values):
    counts = values.dropna().value_counts()
    if counts.empty:
        return None
    top_count = counts.max()
    tied = counts[counts == top_count].index
    return next(r for r in MRECIST_WORST_FIRST if r in tied)

model_mrecist = treatment_arm.merge(model_sen[["model.id", "mRECIST"]], on="model.id", how="left")
mrecist_derived = (
    model_mrecist.groupby("batch.name")["mRECIST"]
    .apply(majority_vote_mrecist)
    .rename("mRECIST")
    .reset_index()
)

n_multi_arm = (treatment_arm.groupby("batch.name").size() > 1).sum()
print(f"Batches with multiple treatment-arm entries (majority vote applies): {n_multi_arm}")

Batches with multiple treatment-arm entries (majority vote applies): 14


### Save cisplatin model list (sensitivity + omics mapping)

In [7]:
cis = cis.merge(omics, on="model_id", how="inner")
cis = cis.merge(n_replicates_derived, on="batch.name", how="left")
cis = cis.merge(mrecist_derived, on="batch.name", how="left")

cis.to_csv(models_out, sep="\t", index=False)
print("Saved model list to:", models_out)

print(cis["mRECIST"].value_counts())

# A model's mRECIST can be missing even though it has CNV/mutation data, if none
# of its treatment-arm model.id entries have a recorded mRECIST in
# model_sensitivity.csv. Kept in the cohort (still has valid omics data), just
# flagged here since it's silently excluded from the value_counts above.
missing_mrecist = cis.loc[cis["mRECIST"].isna(), "model_id"].tolist()
if missing_mrecist:
    print(f"\nModels kept but with no recorded mRECIST: {missing_mrecist}")

Saved model list to: /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study/data/procdata/3_carboplatin/McGill_cisplatin_models.tsv
mRECIST
PD    9
PR    7
SD    7
CR    6
Name: count, dtype: int64


### Report replicate condition

In [8]:
print(f"n_replicates distribution across the {len(cis)} cisplatin models:")
print(cis["n_replicates"].value_counts().sort_index())

n_multi_replicate = (cis["n_replicates"] > 1).sum()
print(f"\nModels with >1 treatment-arm replicate: {n_multi_replicate}/{len(cis)}")

n_replicates distribution across the 29 cisplatin models:
n_replicates
1    29
Name: count, dtype: int64

Models with >1 treatment-arm replicate: 0/29


### Filter mutation and CNV matrices to cisplatin models

In [9]:
mut = pd.read_csv(mutation_file, index_col=0)
cnv = pd.read_csv(cnv_file, index_col=0)
models_ext = cis
print("Mutation matrix shape:", mut.shape)
print("CNV matrix shape:", cnv.shape)

# Rename omics columns (biobase sample IDs) to their model_id
sample_map = dict(zip(models_ext["omics_id"], models_ext["model_id"]))
print("Mapped samples:", len(sample_map))
mut = mut.rename(columns=sample_map)
cnv = cnv.rename(columns=sample_map)

# Keep only the cisplatin models
model_ids = sorted(models_ext["model_id"].unique())
mut = mut[mut.columns.intersection(model_ids)]
cnv = cnv[cnv.columns.intersection(model_ids)]
print("Mutation columns after renaming:", mut.shape[1])
print("CNV columns after renaming:", cnv.shape[1])

# Not every model necessarily has both a mutation-panel and a CNV column;
# keep only models present in both.
common_models = sorted(set(mut.columns) & set(cnv.columns))
print("Models with mutation + CNV:", len(common_models))

# Every model that passed the CNV-omics filter above should also have mutation
# data -- if this ever fails, some model silently lost its mutation column
# during the rename/intersect steps above and needs investigating.
assert len(common_models) == len(model_ids), (
    f"Expected all {len(model_ids)} CNV-bearing cisplatin models to also have "
    f"mutation data, but only {len(common_models)} have both"
)

mut_filtered = mut[common_models]
cnv_filtered = cnv[common_models]
print("Filtered mutation matrix:", mut_filtered.shape)
print("Filtered CNV matrix:", cnv_filtered.shape)

mut_filtered.to_csv(mut_out)
cnv_filtered.to_csv(cnv_out)
print("Saved:", mut_out)
print("Saved:", cnv_out)

Mutation matrix shape: (4876, 70)
CNV matrix shape: (27090, 70)
Mapped samples: 29
Mutation columns after renaming: 29
CNV columns after renaming: 29
Models with mutation + CNV: 29
Filtered mutation matrix: (4876, 29)
Filtered CNV matrix: (27090, 29)
Saved: /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study/data/procdata/3_carboplatin/McGill_mutation_cisplatin_filtered.csv
Saved: /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study/data/procdata/3_carboplatin/McGill_cnv_cisplatin_filtered.csv


### Prepare NeST-VNN inputs: load target gene list

In [10]:
# Load in nesv_vnn genes
gene_map = pd.read_csv(gene_file, sep="\t", header=None)
gene_map.to_csv(gene_vnn, sep="\t", header=False, index=False)
gene_map.columns = ["index", "gene"]
gene_list = gene_map["gene"].tolist()
print("NEST_VNN Gene:", len(gene_list))

# Restrict to the NeST-VNN target genes
mut = mut_filtered[mut_filtered.index.isin(gene_list)]
cnv = cnv_filtered[cnv_filtered.index.isin(gene_list)]
print("After NeST gene filter -- Mutation:", mut.shape, "| CNV:", cnv.shape)

NEST_VNN Gene: 718
After NeST gene filter -- Mutation: (235, 29) | CNV: (717, 29)


### Binarize mutation and CNV calls

In [11]:
# Mutation: 1 if any functional variant classification is present, else 0
mut_bin = mut.map(binarize_mutation_inclusive)
mut_bin = mut_bin.reindex(gene_list).fillna(0)

# CNV: McGill uses GISTIC-style discrete calls in {-2,-1,0,1,2}
# (verified against CNV_matrix.csv values) -- 2 = amplification, -2 = deletion
cna = (cnv == 2).astype(int)
cnd = (cnv == -2).astype(int)
cna = cna.reindex(gene_list).fillna(0)
cnd = cnd.reindex(gene_list).fillna(0)

print("Mutation calls:", int(mut_bin.to_numpy().sum()))
print("Amplification calls:", int(cna.to_numpy().sum()))
print("Deletion calls:", int(cnd.to_numpy().sum()))

Mutation calls: 236
Amplification calls: 2122
Deletion calls: 1452


### Reorient to cell x gene format and save NeST-VNN input files

In [12]:
# NeST-VNN expects rows = cells (models), columns = genes in gene_list order
mut_bin = mut_bin.T.astype(int)
cna = cna.T.astype(int)
cnd = cnd.T.astype(int)

mut_bin.to_csv(mut_vnn, header=False, index=False)
cna.to_csv(cna_vnn, header=False, index=False)
cnd.to_csv(cnd_vnn, header=False, index=False)

# cell2ind.txt row order must match the matrices above
cells = mut_bin.index.tolist()
cell2ind = pd.DataFrame({"index": range(len(cells)), "cell": cells})
cell2ind.to_csv(cell_vnn, sep="\t", header=False, index=False)

# test_data.txt: placeholder AUC (0), prediction-only run.
# "drug"/"model" values aren't used for lookup by NeST-VNN's predict.py
# (verified against ml_models/nest_vnn/src/util.py); only "dataset" and
# the cell2id mapping matter for the actual predictions.
test = pd.DataFrame({"cell": cells, "drug": "Cisplatin", "test": 0, "model": "PDX"})
test.to_csv(test_data_vnn, sep="\t", header=False, index=False)

print("Mutation matrix:", mut_bin.shape)
print("CNA matrix:", cna.shape)
print("CND matrix:", cnd.shape)

Mutation matrix: (29, 718)
CNA matrix: (29, 718)
CND matrix: (29, 718)


### Matrix statistics (QC)

In [13]:
matrix_stats("Mutation", mut_bin)
matrix_stats("CNA", cna)
matrix_stats("CND", cnd)


Mutation statistics
----------------------------
Models: 29
Genes: 718
Total entries: 20822
Nonzero entries: 236
Sparsity: 0.9887

Per model events
Median: 7.0
Mean: 8.14
Max: 38

Per gene events
Median: 0.0
Mean: 0.33
Max: 23

CNA statistics
----------------------------
Models: 29
Genes: 718
Total entries: 20822
Nonzero entries: 2122
Sparsity: 0.8981

Per model events
Median: 68.0
Mean: 73.17
Max: 148

Per gene events
Median: 2.0
Mean: 2.96
Max: 20

CND statistics
----------------------------
Models: 29
Genes: 718
Total entries: 20822
Nonzero entries: 1452
Sparsity: 0.9303

Per model events
Median: 26.0
Mean: 50.07
Max: 213

Per gene events
Median: 1.0
Mean: 2.02
Max: 10
